# Skin Disease Classifier - Colab Training & Prediction

Upload your `disease.zip` file to Colab and run the cells in order. This notebook trains a simple image classifier and lets you predict from a new image.

## Step 1: Upload or confirm the dataset zip file

In [ ]:
import glob
from google.colab import files
import os

zip_files = glob.glob("*.zip")
if not zip_files:
    print("No .zip file found. Please upload your dataset zip file.")
    uploaded = files.upload()
    zip_files = list(uploaded.keys())
if not zip_files:
    raise FileNotFoundError("No dataset zip file uploaded.")
zip_path = "disease.zip" if "disease.zip" in zip_files else zip_files[0]
print("Using dataset zip:", zip_path)
with open("zip_path.txt", "w") as f:
    f.write(zip_path)


## Step 2: Extract the dataset zip and verify folders

In [ ]:
import os
import zipfile

with open("zip_path.txt", "r") as f:
    zip_path = f.read().strip()
extract_path = "/content/skin_data"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

dataset_root = None
for root, dirs, files in os.walk(extract_path):
    if "train" in dirs and "test" in dirs:
        dataset_root = root
        break
assert dataset_root is not None, "Could not find train/ and test/ folders inside the zip."
train_dir = os.path.join(dataset_root, "train")
test_dir = os.path.join(dataset_root, "test")
print("Dataset root:", dataset_root)
print("Train folder:", train_dir)
print("Test folder:", test_dir)
classes = sorted(os.listdir(train_dir))
print("Found classes:", classes)
for cls in classes:
    cls_path = os.path.join(train_dir, cls)
    print(f"  {cls}: {len(os.listdir(cls_path))} images")


## Step 3: Load the dataset into TensorFlow

In [ ]:
import tensorflow as tf
img_size = 224
batch_size = 32
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_size, img_size),
    batch_size=batch_size,
    label_mode="int"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(img_size, img_size),
    batch_size=batch_size,
    label_mode="int"
)
class_names = train_ds.class_names
print("class_names:", class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## Step 4: Build a simple CNN model

In [ ]:
from tensorflow.keras import layers, models
num_classes = len(class_names)
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(img_size, img_size, 3)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),
])
model.summary()


## Step 5: Compile and train

In [ ]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_ds, validation_data=test_ds, epochs=10)


## Step 6: Evaluate the model

In [ ]:
loss, accuracy = model.evaluate(test_ds)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy*100:.2f}%")


## Step 7: Save the model and labels

In [ ]:
import json
model.save("/content/skin_model.h5")
with open("/content/class_names.json", "w") as f:
    json.dump(class_names, f)
print("Saved /content/skin_model.h5 and /content/class_names.json")


## Step 8: Predict a new image (chatbot style)

In [25]:
import numpy as np
from tensorflow.keras.preprocessing import image
import os
image_path = input("Enter a local image path to predict: ").strip()
image_path = input("Enter a local image path to predict: ").strip()
image_path = image_path.strip('"').strip("'")
assert os.path.exists(image_path), f"File not found: {image_path}"
img = image.load_img(image_path, target_size=(img_size, img_size))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, 0)
pred = model.predict(img_array)[0]
top = pred.argsort()[-3:][::-1]
print("Bot: Top predictions:")
for i in top:
    print(f" - {class_names[int(i)]}: {pred[int(i)]*100:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Bot: Top predictions:
 - Unknown_Normal: 53.82%
 - Vitiligo: 29.47%
 - Bullous: 4.90%


## Step 9: Download the trained model and labels

In [18]:
try:
    from google.colab import files
except ImportError:
    files = None

import os
print('Preparing downloads for your model files...')
model_path = '/content/skin_model.h5'
labels_path = '/content/class_names.json'
if files is not None and os.path.exists(model_path):
    files.download(model_path)
else:
    print(f'Could not download {model_path}.')
    print('If you are not in Google Colab, retrieve the files from the notebook storage.')
if files is not None and os.path.exists(labels_path):
    files.download(labels_path)
else:
    print(f'Could not download {labels_path}.')
print('✓ Download step completed!')


Preparing downloads for your model files...
Could not download /content/skin_model.h5.
If you are not in Google Colab, retrieve the files from the notebook storage.
Could not download /content/class_names.json.
✓ Download step completed!
